# RealWaste SAM Segmentation Preparation

Standalone Kaggle notebook for preparing RealWaste into the same normalized COCO-like format used by the merge pipeline.

The notebook discovers a class-folder RealWaste dataset under `/kaggle/input`, maps RealWaste classes into the AquaTrash 8-class label space, downloads a high-accuracy SAM checkpoint, uses it to synthesize one polygon mask per image, and writes outputs to `/kaggle/working/data/normalized/realwaste`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

INSTALL_MISSING_PACKAGES = True
if INSTALL_MISSING_PACKAGES:
    required_packages = [
        ("ultralytics", "ultralytics>=8.0"),
        ("PIL", "pillow>=10.0"),
        ("matplotlib", "matplotlib>=3.7"),
    ]
    missing_packages = [package for module, package in required_packages if importlib.util.find_spec(module) is None]
    if missing_packages:
        print("Installing missing packages:", missing_packages)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Dependencies are already installed.")

In [ ]:
from PIL import Image

IS_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path("data/raw")
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")

# Leave as None to auto-discover a folder containing RealWaste class directories.
REALWASTE_ROOT_OVERRIDE = None

# High-accuracy SAM checkpoint supported by Ultralytics assets.
# If Kaggle GPU memory is tight, switch to "sam2.1_b.pt" or "sam_l.pt"; for speed, use "mobile_sam.pt".
SAM_MODEL = "sam2.1_l.pt"
SAM_WEIGHTS_DIR = WORKING_DIR / "models" / "sam"
SAM_IMGSZ = 1024
SAM_POINTS = "center_plus_quarters"
FORCE_REGENERATE_MASKS = False
SAVE_CACHE_EVERY = 25
MAX_IMAGES = None

OUTPUT_DIR = WORKING_DIR / "data" / "normalized" / "realwaste"
CACHE_PATH = OUTPUT_DIR / "sam_segmentations_cache.json"
ANNOTATIONS_PATH = OUTPUT_DIR / "annotations.json"
SUMMARY_PATH = OUTPUT_DIR / "realwaste_prepare_summary.json"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_CLASSES = {
    "Cardboard",
    "Food Organics",
    "Glass",
    "Metal",
    "Miscellaneous Trash",
    "Paper",
    "Plastic",
    "Textile Trash",
    "Vegetation",
}

AQUATRASH_CATEGORIES = [
    {"id": 1, "name": "glass"},
    {"id": 2, "name": "metal_can"},
    {"id": 3, "name": "paper_cardboard"},
    {"id": 4, "name": "plastic_battle"},
    {"id": 5, "name": "plastic_bag"},
    {"id": 6, "name": "rigid_plastic"},
    {"id": 7, "name": "organic_waste"},
    {"id": 8, "name": "mixed_waste"},
]

REALWASTE_TO_AQUATRASH_LABEL = {
    "Cardboard": "paper_cardboard",
    "Food Organics": "organic_waste",
    "Glass": "glass",
    "Metal": "metal_can",
    "Miscellaneous Trash": "mixed_waste",
    "Paper": "paper_cardboard",
    "Plastic": "rigid_plastic",
    "Textile Trash": "mixed_waste",
    "Vegetation": "organic_waste",
}

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORKING_DIR:", WORKING_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SAM_MODEL:", SAM_MODEL)
print("SAM_WEIGHTS_DIR:", SAM_WEIGHTS_DIR)
print("SAM_IMGSZ:", SAM_IMGSZ)

if INPUT_ROOT.exists():
    print("\nInput roots:")
    for path in sorted(INPUT_ROOT.iterdir()):
        print(" -", path)

In [ ]:
# Download and load the SAM checkpoint before the image-processing loop starts.
# This catches unsupported checkpoint names before the long RealWaste processing cell runs.
from ultralytics import SAM
from ultralytics.utils.downloads import ASSETS_URL, GITHUB_ASSETS_NAMES, safe_download

SAM_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

sam_model_path = Path(SAM_MODEL)
if sam_model_path.exists():
    SAM_MODEL_PATH = sam_model_path.resolve()
elif SAM_MODEL in GITHUB_ASSETS_NAMES:
    SAM_MODEL_PATH = SAM_WEIGHTS_DIR / SAM_MODEL
    if not SAM_MODEL_PATH.exists():
        print(f"Downloading SAM checkpoint {SAM_MODEL} to {SAM_MODEL_PATH}...")
        safe_download(
            url=f"{ASSETS_URL}/{SAM_MODEL}",
            file=SAM_MODEL_PATH,
            min_bytes=1_000_000,
            exist_ok=True,
            progress=True,
        )
    else:
        print(f"SAM checkpoint already exists: {SAM_MODEL_PATH}")
else:
    supported = sorted(name for name in GITHUB_ASSETS_NAMES if "sam" in name.lower())
    raise ValueError(
        f"Unsupported SAM_MODEL '{SAM_MODEL}'. Supported Ultralytics SAM assets include: {supported}. "
        "Set SAM_MODEL to one of these names or to a local .pt/.pth path."
    )

print(f"Loading SAM model checkpoint: {SAM_MODEL_PATH}")
SAM_MODEL_INSTANCE = SAM(str(SAM_MODEL_PATH))
try:
    SAM_MODEL_INSTANCE.info(verbose=True)
except Exception as exc:
    print(f"SAM model loaded, but model info could not be printed: {type(exc).__name__}: {exc}")
print("SAM model is ready.")


In [ ]:
def discover_realwaste_root(input_root: Path = INPUT_ROOT, override: str | Path | None = REALWASTE_ROOT_OVERRIDE) -> Path:
    if override:
        root = Path(override)
        if not root.exists():
            raise FileNotFoundError(f"REALWASTE_ROOT_OVERRIDE does not exist: {root}")
        return root

    search_roots = [input_root]
    if not IS_KAGGLE:
        search_roots.extend([Path("data/raw"), Path("../data/raw")])

    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        paths = [root]
        paths.extend(path for path in root.glob("*") if path.is_dir())
        paths.extend(path for path in root.glob("*/*") if path.is_dir())
        for path in paths:
            try:
                child_dirs = {child.name for child in path.iterdir() if child.is_dir()}
            except OSError:
                continue
            score = len(child_dirs & EXPECTED_CLASSES)
            if score >= 5:
                candidates.append((score, -len(path.parts), path))

    if not candidates:
        raise FileNotFoundError("Could not find a RealWaste root containing the expected class folders.")

    candidates.sort(reverse=True)
    return candidates[0][2]


def safe_file_component(value: str) -> str:
    return "_".join(value.replace("-", "_").split())


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def polygon_area(segmentation: list[float]) -> float:
    if len(segmentation) < 6:
        return 0.0
    points = list(zip(segmentation[0::2], segmentation[1::2]))
    area = 0.0
    for index, (x1, y1) in enumerate(points):
        x2, y2 = points[(index + 1) % len(points)]
        area += (x1 * y2) - (x2 * y1)
    return abs(area) / 2.0


def bbox_from_segmentation(segmentation: list[float]) -> list[float]:
    xs = segmentation[0::2]
    ys = segmentation[1::2]
    min_x = min(xs)
    min_y = min(ys)
    max_x = max(xs)
    max_y = max(ys)
    return [min_x, min_y, max_x - min_x, max_y - min_y]


def rectangle_segmentation(width: int, height: int) -> list[float]:
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    return [0.0, 0.0, max_x, 0.0, max_x, max_y, 0.0, max_y]


def clamp_polygon(segmentation: list[float], width: int, height: int) -> list[float]:
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    clamped = []
    for index in range(0, len(segmentation) - 1, 2):
        x = max(0.0, min(max_x, float(segmentation[index])))
        y = max(0.0, min(max_y, float(segmentation[index + 1])))
        clamped.extend([x, y])
    return clamped


def prompt_points(width: int, height: int) -> list[list[float]]:
    if SAM_POINTS == "center":
        return [[width / 2.0, height / 2.0]]
    return [
        [width / 2.0, height / 2.0],
        [width * 0.38, height / 2.0],
        [width * 0.62, height / 2.0],
        [width / 2.0, height * 0.38],
        [width / 2.0, height * 0.62],
    ]


def load_sam_model():
    global SAM_MODEL_INSTANCE
    if "SAM_MODEL_INSTANCE" not in globals() or SAM_MODEL_INSTANCE is None:
        from ultralytics import SAM

        model_path = globals().get("SAM_MODEL_PATH", Path(SAM_MODEL))
        print(f"Loading SAM model checkpoint: {model_path}")
        SAM_MODEL_INSTANCE = SAM(str(model_path))
    return SAM_MODEL_INSTANCE


def generate_sam_segmentation(sam_model, image_path: Path, width: int, height: int) -> tuple[list[float], bool]:
    points = prompt_points(width, height)
    results = sam_model.predict(
        source=str(image_path),
        points=points,
        labels=[1] * len(points),
        imgsz=SAM_IMGSZ,
        retina_masks=True,
        verbose=False,
        save=False,
    )

    polygons = []
    if results and getattr(results[0], "masks", None) is not None:
        for polygon in results[0].masks.xy:
            segmentation = []
            for x, y in polygon:
                segmentation.extend([float(x), float(y)])
            segmentation = clamp_polygon(segmentation, width, height)
            area = polygon_area(segmentation)
            area_ratio = area / float(max(width * height, 1))
            if len(segmentation) >= 6 and 0.001 <= area_ratio <= 0.995:
                polygons.append((area, segmentation))

    if not polygons:
        return rectangle_segmentation(width, height), False

    polygons.sort(key=lambda item: item[0], reverse=True)
    return polygons[0][1], True

In [ ]:
def collect_realwaste_images(root_dir: Path) -> list[tuple[str, Path]]:
    image_paths = []
    for class_name in sorted(EXPECTED_CLASSES):
        class_dir = root_dir / class_name
        if not class_dir.exists():
            print(f"Warning: class folder not found: {class_dir}")
            continue
        for image_path in sorted(class_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append((class_name, image_path))
    if MAX_IMAGES is not None:
        image_paths = image_paths[:MAX_IMAGES]
    return image_paths


def prepare_realwaste_with_sam(root_dir: Path) -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    label_to_id = {category["name"]: category["id"] for category in AQUATRASH_CATEGORIES}
    cache = load_json(CACHE_PATH)
    image_paths = collect_realwaste_images(root_dir)
    if not image_paths:
        raise FileNotFoundError(f"No RealWaste images found under {root_dir}")

    print(f"Preparing {len(image_paths)} RealWaste images from {root_dir}")
    print(f"SAM cache: {CACHE_PATH}")
    sam_model = None
    images = []
    annotations = []
    counts_by_class = {class_name: 0 for class_name in sorted(EXPECTED_CLASSES)}
    fallback_masks = 0

    for image_index, (class_name, image_path) in enumerate(image_paths, start=1):
        with Image.open(image_path) as image:
            width, height = image.size

        cache_key = image_path.relative_to(root_dir).as_posix()
        cached_mask = cache.get(cache_key)
        if (
            cached_mask is not None
            and not FORCE_REGENERATE_MASKS
            and int(cached_mask.get("width", width)) == width
            and int(cached_mask.get("height", height)) == height
        ):
            segmentation = cached_mask["segmentation"]
            used_sam = bool(cached_mask.get("used_sam", True))
        else:
            if sam_model is None:
                sam_model = load_sam_model()
            segmentation, used_sam = generate_sam_segmentation(sam_model, image_path, width, height)
            cache[cache_key] = {
                "segmentation": segmentation,
                "used_sam": used_sam,
                "width": width,
                "height": height,
            }

        if not used_sam:
            fallback_masks += 1

        output_file_name = f"{safe_file_component(class_name)}_{image_path.name.replace(' ', '_')}"
        shutil.copy2(image_path, OUTPUT_DIR / output_file_name)

        image_id = len(images) + 1
        annotation_id = len(annotations) + 1
        aquatrash_label = REALWASTE_TO_AQUATRASH_LABEL[class_name]
        category_id = label_to_id[aquatrash_label]
        area = polygon_area(segmentation)
        bbox = bbox_from_segmentation(segmentation)

        images.append({
            "id": image_id,
            "width": width,
            "height": height,
            "file_name": output_file_name,
            "source_file_name": cache_key,
        })
        annotations.append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "segmentation": [segmentation],
            "area": area,
            "bbox": bbox,
            "iscrowd": 0,
        })
        counts_by_class[class_name] += 1

        if image_index % SAVE_CACHE_EVERY == 0:
            save_json(cache, CACHE_PATH)
        if image_index % 100 == 0:
            print(f"Prepared {image_index}/{len(image_paths)} masks")

    save_json(cache, CACHE_PATH)
    payload = {
        "images": images,
        "annotations": annotations,
        "categories": AQUATRASH_CATEGORIES,
        "info": {
            "source": "RealWaste",
            "sam_model": SAM_MODEL,
            "sam_imgsz": SAM_IMGSZ,
            "sam_points": SAM_POINTS,
            "sam_cache": str(CACHE_PATH),
            "fallback_masks": fallback_masks,
        },
    }
    save_json(payload, ANNOTATIONS_PATH)

    summary = {
        "realwaste_root": str(root_dir),
        "output_dir": str(OUTPUT_DIR),
        "annotations_path": str(ANNOTATIONS_PATH),
        "sam_model": SAM_MODEL,
        "images": len(images),
        "annotations": len(annotations),
        "fallback_masks": fallback_masks,
        "counts_by_class": counts_by_class,
        "target_categories": AQUATRASH_CATEGORIES,
    }
    save_json(summary, SUMMARY_PATH)
    return summary

In [ ]:
REALWASTE_ROOT = discover_realwaste_root()
print("REALWASTE_ROOT:", REALWASTE_ROOT)

summary = prepare_realwaste_with_sam(REALWASTE_ROOT)
print(json.dumps(summary, indent=2)[:4000])

In [ ]:
def visualize_prepared_samples(samples: int = 6) -> None:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon

    payload = load_json(ANNOTATIONS_PATH)
    annotations_by_image = {ann["image_id"]: ann for ann in payload.get("annotations", [])}
    categories = {cat["id"]: cat["name"] for cat in payload.get("categories", [])}
    images = payload.get("images", [])[:samples]
    if not images:
        print("No prepared images to visualize.")
        return

    cols = min(3, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")

    for ax, image_info in zip(axes.flat, images):
        image_path = OUTPUT_DIR / image_info["file_name"]
        ann = annotations_by_image.get(image_info["id"])
        image = Image.open(image_path)
        ax.imshow(image)
        ax.set_title(image_info["file_name"], fontsize=8)
        if ann:
            segmentation = ann["segmentation"][0]
            points = list(zip(segmentation[0::2], segmentation[1::2]))
            ax.add_patch(Polygon(points, closed=True, fill=False, edgecolor="yellow", linewidth=1.5))
            ax.text(
                points[0][0],
                points[0][1],
                categories.get(ann["category_id"], str(ann["category_id"])),
                color="black",
                fontsize=8,
                bbox={"facecolor": "yellow", "edgecolor": "none", "pad": 1},
            )

    plt.tight_layout()
    plt.show()


visualize_prepared_samples(samples=6)